In [2]:
from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_2jQL4Z_7fBeRQ3h15rp4Pqvv4sQPrycimZBWCGTfPVr6s6AVSRzwWWYGbpZzM5UDjC1jD4")

index = pc.Index("spring-tutor")

NotFoundError: [404 NOT_FOUND] Resource spring-tutor not found

In [4]:
api_key="pcsk_2jQL4Z_7fBeRQ3h15rp4Pqvv4sQPrycimZBWCGTfPVr6s6AVSRzwWWYGbpZzM5UDjC1jD4"

In [5]:
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)

print(pc.list_indexes())

IndexList([])


In [8]:
from dotenv import load_dotenv
load_dotenv()

False

In [10]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vec = embeddings.embed_query("hello")

print(len(vec))

768


In [12]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=api_key)

pc.create_index(
    name="spring-tutor",
    dimension=768,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

Name:,spring-tutor
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://spring-tutor-zt8qy8d.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [13]:
print(pc.list_indexes())

IndexList([<name='spring-tutor', ready=True>])


In [14]:
index = pc.Index("spring-tutor")

In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_190962/4074419481.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
from pathlib import Path
import re
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = Path("documents")

all_docs = []

for pdf_file in sorted(pdf_folder.glob("*.pdf")):

    match = re.search(r"\((\d+)\)", pdf_file.name)
    lecture_no = int(match.group(1))

    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    for doc in docs:
        doc.metadata["course"] = "Spring Boot"
        doc.metadata["lecture"] = lecture_no
        doc.metadata["source_file"] = pdf_file.name
        doc.metadata["topic"] = doc.metadata.get("title", "Unknown")

    all_docs.extend(docs)

print("Total Pages:", len(all_docs))
print(all_docs[0].metadata)

Total Pages: 1301
{'producer': 'Skia/PDF m128', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'creationdate': '2026-07-24T09:24:42+00:00', 'title': 'Spring JDBC: From Raw JDBC to JdbcTemplate', 'moddate': '2026-07-24T09:24:42+00:00', 'source': 'documents/Notes (1).pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'course': 'Spring Boot', 'lecture': 1, 'source_file': 'Notes (1).pdf', 'topic': 'Spring JDBC: From Raw JDBC to JdbcTemplate'}


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    
    chunk_overlap=300
)

chunks = splitter.split_documents(all_docs)

print("Total Chunks:", len(chunks))

Total Chunks: 1302


In [5]:
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)

index = pc.Index("spring-tutor")

In [6]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [7]:
import uuid

batch_size = 100

for i in range(0, len(chunks), batch_size):

    batch = chunks[i:i + batch_size]

    vectors = []

    for chunk in batch:

        vector = embeddings.embed_query(
            chunk.page_content
        )

        vectors.append({
            "id": str(uuid.uuid4()),
            "values": vector,
            "metadata": {
                "text": chunk.page_content,
                **chunk.metadata
            }
        })

    index.upsert(vectors=vectors)

    print(
        f"Uploaded {min(i + batch_size, len(chunks))}/{len(chunks)}"
    )

Uploaded 100/1302
Uploaded 200/1302
Uploaded 300/1302
Uploaded 400/1302
Uploaded 500/1302
Uploaded 600/1302
Uploaded 700/1302
Uploaded 800/1302
Uploaded 900/1302
Uploaded 1000/1302
Uploaded 1100/1302
Uploaded 1200/1302
Uploaded 1300/1302
Uploaded 1302/1302


In [8]:
query_embedding = embeddings.embed_query(
    "What is Spring Bean?"
)

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True
)

results

QueryResponse(matches=[ScoredVector(id='5476888c-2a30-4dd6-bc7f-ad229d79d966', score=0.727809906, values=[], metadata={'course': 'Spring Boot', 'creationdate': '2026-06-23T09:41:59+00:00', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'lecture': 8, 'moddate': '2026-06-23T09:41:59+00:00', 'page': 29, 'page_label': '30', 'producer': 'Skia/PDF m128', 'source': 'documents/Notes (8).pdf', 'source_file': 'Notes (8).pdf', 'text': 'It is used to run logic after dependency injection.\nIt is part of initialization.\n21. One-Line Takeaway\nSpring Bean Lifecycle is not just object creation. It is the complete process\nthrough which Spring discovers, creates, prepares, manages, and destroys a\nbean.\nSpring Bean Lifecycle\n30\nCoder Army', 'title': 'Spring Bean Lifecycle', 'topic': 'Spring Bean Lifecycle', 'total_pages': 30}), ScoredVector(id='ffbf63fa-2931-4036-b919-cdb15cac77c5', score=0.72591418, values=[], metadata={'